In [ ]:
import os
import datasets
import pandas as pd
import tqdm
import random

dataset_root = '/data/lihaochen/datasets/TikTok_dataset/'

split = range(1, 341)
# random split range into train and test, with 90% for train and 10% for test
train_split = random.sample(split, int(len(split) * 0.9))
test_split = [i for i in split if i not in train_split]
split = {'train': train_split, 'test': test_split}

for s in ['train', 'test']:
    dataset = []
    for j in sorted(split[s]):
        video_dir = os.path.join(dataset_root, '%05d' % j)
        image_dir = os.path.join(video_dir, 'images')
        video_len = len(os.listdir(image_dir))
        caption_file = open(os.path.join(video_dir, 'captions.txt')).read().strip()
        captions = caption_file.split('\n')
        frames = []

        # generate each frame
        for i in range(1, video_len+1):
            person = f'{j}'
            phi = 0
            mask = os.path.join(video_dir, 'masks', f'{i:04d}.png')
            source = os.path.join(video_dir, 'relight', f'{i:04d}.png')
            target = os.path.join(video_dir, 'images', f'{i:04d}.png')
            img_depth = os.path.join(video_dir, 'img_depth', f'{i:04d}.npy')
            bg_depth = os.path.join(video_dir, 'bg_depth.npy')
            lighting = os.path.join(video_dir, 'hdr', 'refined.exr')
            bg = os.path.join(video_dir, 'inpainted', f'{i:04d}.png')
            caption = captions[i-1]
            if not os.path.exists(img_depth):
                continue
            if not os.path.exists(source):
                continue
            frames.append([person, phi, mask, source, target, img_depth, bg_depth, caption, lighting, bg])

        # group frames into sequences of 4, overlapping by 2
        for i in range(0, len(frames), 2):
            if i + 4 > len(frames):
                dataset.append(frames[-4:])
                break
            dataset.append(frames[i:i+4])

    # gather dataset into a dict
    dict_dataset = {
        'person': [[r[0] for r in frames] for frames in dataset],
        'phi': [[r[1] for r in frames] for frames in dataset],
        'mask': [[r[2] for r in frames] for frames in dataset],
        'source': [[r[3] for r in frames] for frames in dataset],
        'target': [[r[4] for r in frames] for frames in dataset],
        'img_depth': [[r[5] for r in frames] for frames in dataset],
        'bg_depth': [[r[6] for r in frames] for frames in dataset],
        'caption': [[r[7] for r in frames] for frames in dataset],
        'lighting': [[r[8] for r in frames] for frames in dataset],
        'bg': [[r[9] for r in frames] for frames in dataset],
    }

    ds = datasets.Dataset.from_dict(dict_dataset)

    ds.to_parquet(f'./video_{s}.parquet')

In [ ]:
# dataset read test
import os
from datasets import load_dataset
from torchvision import transforms
from torchvision.transforms import v2
from torchvision.transforms.v2.functional import crop
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch

ds = load_dataset('parquet', data_files='./video_test.parquet')

os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1" 
os.environ["OPENCV_IMGCODECS_USE_OPENEXR"] = "1"

class TopCenterCrop:
    def __init__(self, resolution):
        self.resolution = resolution

    def __call__(self, img):
        _, height, width = img.shape  # 获取图像宽高
        if height > width:  # 竖图
            top = max(0, height // 4 - self.resolution // 2)  # 向上偏移裁剪
            left = max(0, (width - self.resolution) // 2)  # 水平居中裁剪
        else:  # 横图，保持中心裁剪
            top = max(0, (height - self.resolution) // 2)
            left = max(0, (width - self.resolution) // 2)

        return crop(img, top, left, self.resolution, self.resolution)

image_transforms = v2.Compose(
    [
        v2.ToTensor(),
        v2.Resize(1024, interpolation=v2.InterpolationMode.BILINEAR),
        TopCenterCrop(1024),
        v2.Normalize([0.5], [0.5]),
    ]
)

source_transforms = v2.Compose(
    [
        v2.ToTensor(),
        v2.Resize(1024, interpolation=v2.InterpolationMode.BILINEAR),
        TopCenterCrop(1024),
        v2.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
        v2.Normalize([0.5], [0.5]),
    ]
)

conditioning_image_transforms = v2.Compose(
    [
        v2.ToTensor(),
        v2.Resize(1024, interpolation=v2.InterpolationMode.BILINEAR),
        TopCenterCrop(1024),
    ]
)

bg_image_transforms = v2.Compose(
    [
        v2.ToTensor(),
        v2.Resize(size=32, max_size=64, interpolation=v2.InterpolationMode.BILINEAR),
        v2.CenterCrop(32),
        v2.functional.horizontal_flip,
    ]
)

def adjust_and_fuse_depth(foreground_depth, background_depth, foreground_mask, bottom_rows=5):
    """
    将前景深度图与背景深度图融合，确保前景物体（如人）的脚部深度与背景一致。

    参数:
        foreground_depth (np.ndarray): 前景深度图。
        background_depth (np.ndarray): 背景深度图。
        foreground_mask (np.ndarray): 前景掩码，1表示前景，0表示背景。
        bottom_rows (int): 取前景物体底部的行数，默认为5。

    返回:
        np.ndarray: 融合后的深度图。
    """
    # 确保输入数组的尺寸一致
    assert foreground_depth.shape == background_depth.shape == foreground_mask.shape, "输入数组的尺寸必须一致"
    
    # 找到前景物体的底部区域（取底部多行）
    bottom_mask = np.zeros_like(foreground_mask, dtype=bool)
    rows, cols = np.where(foreground_mask == 1)  # 找到所有前景像素的行和列
    if len(rows) == 0:
        return background_depth  # 如果没有前景物体，直接返回背景深度图
    
    # 找到每一列的前景物体的最底部行
    unique_cols = np.unique(cols)  # 所有有前景物体的列
    for col in unique_cols:
        col_rows = rows[cols == col]  # 当前列的所有前景行
        if len(col_rows) > 0:
            bottom_row = np.max(col_rows)  # 当前列的最底部行
            # 取底部多行
            bottom_mask[col_rows[col_rows >= (bottom_row - bottom_rows + 1)], col] = True
    
    # 计算前景物体底部的平均深度
    foreground_bottom_depth = np.mean(foreground_depth[bottom_mask])
    
    # 计算背景对应区域的平均深度
    background_bottom_depth = np.mean(background_depth[bottom_mask])
    
    # 计算深度差异
    depth_diff = background_bottom_depth - foreground_bottom_depth
    
    # 调整前景物体的深度值
    adjusted_foreground_depth = foreground_depth.copy()
    adjusted_foreground_depth[foreground_mask == 1] += depth_diff
    
    # 将调整后的前景深度信息融合到背景深度图中
    fused_depth = background_depth.copy()
    fused_depth[foreground_mask == 1] = adjusted_foreground_depth[foreground_mask == 1]
    
    return fused_depth

def preprocess_train(examples):
    f = len(examples['source'])
    bs = len(examples['source'][0])

    source = []
    for sources in examples['source']:
        for s in sources:
            source.append(cv2.imread(s, cv2.IMREAD_UNCHANGED))
    source = [cv2.cvtColor(s, cv2.COLOR_BGR2RGB) for s in source]
    source = [source_transforms(s) for s in source]

    target = []
    for targets in examples['target']:
        for t in targets:
            target.append(cv2.imread(t, cv2.IMREAD_UNCHANGED))
    target = [cv2.cvtColor(t, cv2.COLOR_BGR2RGB) for t in target]
    target = [image_transforms(t) for t in target]

    mask = []
    for masks in examples['mask']:
        for m in masks:
            mask.append(cv2.imread(m, cv2.IMREAD_UNCHANGED))

    img_depth = []
    for img_depths in examples['img_depth']:
        for d in img_depths:
            img_depth.append(np.load(d))

    bg_depth = []
    for bg_depths in examples['bg_depth']:
        for d in bg_depths:
            bg_depth.append(np.load(d))

    depth = [adjust_and_fuse_depth(img, bg, m) for img, bg, m in zip(img_depth, bg_depth, mask)]
    
    mask = [np.expand_dims(m, axis=-1) for m in mask]
    mask = [conditioning_image_transforms(m) for m in mask]

    depth = [np.expand_dims(d, axis=-1) for d in depth]
    depth = [conditioning_image_transforms(d) for d in depth]

    lighting = []
    for lightings in examples['lighting']:
        for l in lightings:
            lighting.append(cv2.imread(l, cv2.IMREAD_UNCHANGED))
    lighting = [cv2.cvtColor(l, cv2.COLOR_BGR2RGB) for l in lighting]
    lighting = [np.roll(l, l.shape[1] // 2, 1) for l in lighting]
    lighting = [bg_image_transforms(l) for l in lighting]

    bg = []
    for bgs in examples['bg']:
        for b in bgs:
            bg.append(cv2.imread(b, cv2.IMREAD_UNCHANGED))
    bg = [cv2.cvtColor(b, cv2.COLOR_BGR2RGB) for b in bg]
    bg = [image_transforms(b) for b in bg]

    # group by batch size
    source = [source[i:i+bs] for i in range(0, len(source), bs)]
    target = [target[i:i+bs] for i in range(0, len(target), bs)]
    mask = [mask[i:i+bs] for i in range(0, len(mask), bs)]
    depth = [depth[i:i+bs] for i in range(0, len(depth), bs)]
    lighting = [lighting[i:i+bs] for i in range(0, len(lighting), bs)]
    bg = [bg[i:i+bs] for i in range(0, len(bg), bs)]

    # stack along dim 1
    source = [torch.stack(s, dim=1) for s in source]
    target = [torch.stack(t, dim=1) for t in target]
    mask = [torch.stack(m, dim=1) for m in mask]
    depth = [torch.stack(d, dim=1) for d in depth]
    lighting = [torch.stack(l, dim=1) for l in lighting]
    bg = [torch.stack(b, dim=1) for b in bg]

    examples['source'] = source
    examples['target'] = target
    examples['mask'] = mask
    examples['depth'] = depth
    examples['lighting'] = lighting
    examples['bg'] = bg

    return examples

def collate_fn(examples):
    source = torch.stack([example["source"] for example in examples])
    source = source.to(memory_format=torch.contiguous_format).float()

    target = torch.stack([example["target"] for example in examples])
    target = target.to(memory_format=torch.contiguous_format).float()

    mask = torch.stack([example["mask"] for example in examples])
    mask = mask.to(memory_format=torch.contiguous_format).float()

    depth = torch.stack([example["depth"] for example in examples])
    depth = depth.to(memory_format=torch.contiguous_format).float()

    lighting = torch.stack([example["lighting"] for example in examples])
    lighting = lighting.to(memory_format=torch.contiguous_format).float()

    bg = torch.stack([example["bg"] for example in examples])
    bg = bg.to(memory_format=torch.contiguous_format).float()

    caption = [example["caption"] for example in examples]

    return {
        "source": source,
        "target": target,
        "mask": mask,
        "depth": depth,
        "lighting": lighting,
        "bg": bg,
        "caption": caption,
    }

# dataset = dataset.with_transform(preprocess_train)

ds = ds.with_transform(preprocess_train)
ds = ds['train']
ds = ds.remove_columns(["person", "phi"])

train_dataloader = torch.utils.data.DataLoader(
    ds,
    shuffle=False,
    collate_fn=collate_fn,
    batch_size=2,
    num_workers=0,
)
for idx, sample in enumerate(train_dataloader):
    if idx == 0:
        print(sample['caption'])
    break

/data1/lihaochen/envs/relight/lib/python3.11/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


['A young man with tousled brown hair, wearing a matching powder blue athletic tracksuit, captured in a dynamic, slightly blurred motion as if mid-dance. The setting is a dimly lit bedroom with a cluttered bookshelf visible in the background. Soft, diffused light casts subtle shadows, creating a casual, intimate mood. Focus on capturing the energy and youthful exuberance of the moment, with a slight grain to mimic a candid snapshot. The overall aesthetic should be raw and authentic, reminiscent of a TikTok video still.', 'A young man with tousled brown hair, wearing a faded blue athletic tracksuit, captured in a dynamic, slightly distorted, low-angle shot reminiscent of a TikTok video. The setting is a dimly lit, cluttered bedroom with visible bookshelves. The mood is energetic and playful, with a sense of youthful exuberance. Soft, diffused lighting casts long shadows, emphasizing the movement and creating a slightly grainy, nostalgic aesthetic. Focus on capturing the raw, unfiltered 